In [1]:
# =============
# 环境、路径与训练参数
# =============

import os
import json
import time
import random
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

CURRENT_DIR = Path(".").resolve()
PROJECT_ROOT = Path("..").resolve()

IMAGE_DIR = PROJECT_ROOT / "images"

SOURCE_SPLIT_DIR = PROJECT_ROOT / "fire_splits"
SOURCE_ENSEMBLE_DIR = SOURCE_SPLIT_DIR / "ensemble_splits"

LOCAL_RESULT_DIR = CURRENT_DIR
MODEL_SAVE_DIR = LOCAL_RESULT_DIR / "models"
TRAIN_RESULT_DIR = LOCAL_RESULT_DIR / "train_results"

MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_RESULT_DIR.mkdir(parents=True, exist_ok=True)

VAL_CSV = SOURCE_SPLIT_DIR / "val_fixed_real_only.csv"
TEST_CSV = SOURCE_SPLIT_DIR / "test_fixed_real_only.csv"

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_ENSEMBLE = 1

GLOBAL_IMG_SIZE = 512
TILE_IMG_SIZE = 384

N_TRAIN_TILES = 8
N_EVAL_TILES = 16
POS_TILES_PER_FIRE = 4

EVAL_TILE_SIZE = 512
EVAL_STRIDE = 256
LOCAL_TILE_SIZES = [256, 384, 512]

BATCH_SIZE = 4
NUM_WORKERS = 0

FINETUNE_MODE = "layer4_last"
EPOCHS = 15

LR_HEAD = 1e-5
LR_BACKBONE = 1e-7
WEIGHT_DECAY = 1e-2

POS_WEIGHT = 1.0
THRESHOLD = 0.6
LABEL_SMOOTHING = 0.03
GRAD_CLIP_NORM = 1.0

DROPOUT = 0.35
MAX_POOL_SCALE = 0.25

LOCAL_TOPK = 3
GLOBAL_WEIGHT = 0.15
LOCAL_WEIGHT = 0.85

FINAL_LOSS_WEIGHT = 1.00
GLOBAL_LOSS_WEIGHT = 0.05
TILE_LOSS_WEIGHT = 1.00
GLOBAL_ATTN_LOSS_WEIGHT = 0.10
TILE_ATTN_LOSS_WEIGHT = 0.20
NO_FIRE_BRIGHT_ATTN_WEIGHT = 0.02

ATTN_BOX_EXPAND_RATIO = 0.20
ATTN_MASK_DILATE = 1
BRIGHT_PERCENTILE = 90

MODEL_PREFIX = "fire_global_local_attn_convnext_tiny"

USE_PRETRAINED = True

# =============
# 缓存参数
# =============

CACHE_MODE = "disk"       # "none" / "disk" / "memory"
REBUILD_CACHE = False      # 修改 mask / tile 策略后，第一次运行设为 True；生成完成后改为 False
LOAD_DISK_CACHE_TO_MEMORY = False

TRAIN_TILE_POOL_SIZE = 24
TRAIN_POS_TILE_POOL_SIZE = 12
TRAIN_NEG_TILE_POOL_SIZE = 12
TRAIN_NO_FIRE_TILE_POOL_SIZE = 24
CACHE_BRIGHT_CANDIDATE_NUM = 24

CACHE_VERSION = (
    f"v2_attn_global{GLOBAL_IMG_SIZE}_"
    f"tile{TILE_IMG_SIZE}_"
    f"pool{TRAIN_TILE_POOL_SIZE}_"
    f"eval{N_EVAL_TILES}"
)

CACHE_ROOT = SOURCE_SPLIT_DIR / "tile_cache" / CACHE_VERSION

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.benchmark = True
plt.ioff()

print("DEVICE:", DEVICE)
print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:", IMAGE_DIR)
print("SOURCE_SPLIT_DIR:", SOURCE_SPLIT_DIR)
print("SOURCE_ENSEMBLE_DIR:", SOURCE_ENSEMBLE_DIR)
print("MODEL_SAVE_DIR:", MODEL_SAVE_DIR)
print("TRAIN_RESULT_DIR:", TRAIN_RESULT_DIR)
print("CACHE_MODE:", CACHE_MODE)
print("REBUILD_CACHE:", REBUILD_CACHE)
print("CACHE_ROOT:", CACHE_ROOT)

DEVICE: cuda
CURRENT_DIR: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案
PROJECT_ROOT: E:\Programming\Python\DeepLearning\比赛
IMAGE_DIR: E:\Programming\Python\DeepLearning\比赛\images
SOURCE_SPLIT_DIR: E:\Programming\Python\DeepLearning\比赛\fire_splits
SOURCE_ENSEMBLE_DIR: E:\Programming\Python\DeepLearning\比赛\fire_splits\ensemble_splits
MODEL_SAVE_DIR: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\models
TRAIN_RESULT_DIR: E:\Programming\Python\DeepLearning\比赛\ConvNeXt 方案\train_results
CACHE_MODE: disk
REBUILD_CACHE: False
CACHE_ROOT: E:\Programming\Python\DeepLearning\比赛\fire_splits\tile_cache\v2_attn_global512_tile384_pool24_eval16


In [2]:
# =============
# 检查必要文件
# =============

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"未找到图片目录: {IMAGE_DIR}")

if not SOURCE_SPLIT_DIR.exists():
    raise FileNotFoundError(f"未找到划分目录: {SOURCE_SPLIT_DIR}")

if not SOURCE_ENSEMBLE_DIR.exists():
    raise FileNotFoundError(f"未找到 ensemble 划分目录: {SOURCE_ENSEMBLE_DIR}")

if not VAL_CSV.exists():
    raise FileNotFoundError(f"未找到验证集 CSV: {VAL_CSV}")

if not TEST_CSV.exists():
    print("警告：未找到测试集 CSV:", TEST_CSV)

In [3]:
# =============
# 模型定义：ConvNeXt Attention + Global Local Tile
# =============

class ConvNeXtAttentionFireClassifier(nn.Module):
    def __init__(self, dropout=0.35, use_pretrained=True, max_pool_scale=0.25):
        super().__init__()

        if use_pretrained:
            convnext = models.convnext_tiny(
                weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
            )
        else:
            convnext = models.convnext_tiny(weights=None)

        self.backbone = convnext.features
        self.feature_dim = 768
        self.max_pool_scale = 1.0 if max_pool_scale is None else float(max_pool_scale)

        self.attention = nn.Sequential(
            nn.Conv2d(self.feature_dim, 256, kernel_size=1),
            nn.GELU(),
            nn.Dropout2d(p=0.10),
            nn.Conv2d(256, 1, kernel_size=1)
        )

        fusion_dim = self.feature_dim * 3

        self.fc = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(p=dropout),

            nn.Linear(fusion_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(p=0.35),

            nn.Linear(512, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(p=0.25),

            nn.Linear(128, 1)
        )

    def forward_features(self, x):
        return self.backbone(x)

    def get_attention_map(self, feature_map):
        attn_logits = self.attention(feature_map)
        B, _, H, W = attn_logits.shape
        attention_map = torch.softmax(
            attn_logits.flatten(2),
            dim=-1
        ).reshape(B, 1, H, W)
        return attention_map

    def attention_pooling(self, feature_map):
        B, C, H, W = feature_map.shape
        attn_logits = self.attention(feature_map).flatten(1)
        attn_weights = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        tokens = feature_map.flatten(2).transpose(1, 2)
        attention_feature = (tokens * attn_weights).sum(dim=1)
        return attention_feature

    def forward(self, x, return_attention=False):
        feature_map = self.forward_features(x)

        attn_feature = self.attention_pooling(feature_map)
        avg_feature = F.adaptive_avg_pool2d(feature_map, output_size=1).flatten(1)

        max_feature = F.adaptive_max_pool2d(feature_map, output_size=1).flatten(1)
        max_feature = max_feature * self.max_pool_scale

        fused_feature = torch.cat(
            [attn_feature, avg_feature, max_feature],
            dim=1
        )

        logit = self.fc(fused_feature)

        if return_attention:
            attention_map = self.get_attention_map(feature_map)
            return logit, attention_map

        return logit


def set_finetune_mode(model, finetune_mode="layer4_last"):
    for param in model.parameters():
        param.requires_grad = False

    for param in model.attention.parameters():
        param.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    if finetune_mode == "head":
        pass

    elif finetune_mode == "layer4_last":
        for param in model.backbone[7][-1].parameters():
            param.requires_grad = True

    elif finetune_mode == "layer4_all":
        for param in model.backbone[7].parameters():
            param.requires_grad = True

    elif finetune_mode == "all":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError(f"Unknown finetune_mode: {finetune_mode}")

    return model


class GlobalLocalFireClassifier(nn.Module):
    def __init__(
        self,
        dropout=0.35,
        use_pretrained=True,
        max_pool_scale=0.25,
        topk=3,
        global_weight=0.15,
        local_weight=0.85
    ):
        super().__init__()

        self.topk = int(topk)
        self.global_weight = float(global_weight)
        self.local_weight = float(local_weight)

        self.shared_classifier = ConvNeXtAttentionFireClassifier(
            dropout=dropout,
            use_pretrained=use_pretrained,
            max_pool_scale=max_pool_scale
        )

    def _masked_topk_mean(self, tile_logits, tile_mask):
        B, N = tile_logits.shape
        local_logits = []

        for b in range(B):
            valid = tile_mask[b].bool()
            vals = tile_logits[b][valid]

            if vals.numel() == 0:
                vals = tile_logits[b][:1]

            k = min(self.topk, vals.numel())
            top_vals = torch.topk(vals, k=k, largest=True).values
            local_logits.append(top_vals.mean())

        return torch.stack(local_logits, dim=0)

    def forward(self, global_image, tiles, tile_mask=None, return_attention=True):
        B, N, C, H, W = tiles.shape

        if tile_mask is None:
            tile_mask = torch.ones(B, N, dtype=torch.bool, device=tiles.device)
        else:
            tile_mask = tile_mask.bool()

        global_logit, global_attention = self.shared_classifier(
            global_image,
            return_attention=True
        )
        global_logit = global_logit.squeeze(1)

        flat_tiles = tiles.reshape(B * N, C, H, W)
        flat_tile_logits, flat_tile_attn = self.shared_classifier(
            flat_tiles,
            return_attention=True
        )

        tile_logits = flat_tile_logits.squeeze(1).reshape(B, N)

        _, _, Ah, Aw = flat_tile_attn.shape
        tile_attentions = flat_tile_attn.reshape(B, N, 1, Ah, Aw)

        local_logit = self._masked_topk_mean(
            tile_logits=tile_logits,
            tile_mask=tile_mask
        )

        final_logit = (
            self.global_weight * global_logit +
            self.local_weight * local_logit
        )

        return {
            "logit": final_logit.unsqueeze(1),
            "global_logit": global_logit.unsqueeze(1),
            "local_logit": local_logit.unsqueeze(1),
            "tile_logits": tile_logits,
            "global_attention": global_attention,
            "tile_attentions": tile_attentions
        }


def build_global_local_fire_model(
    finetune_mode="layer4_last",
    use_pretrained=True,
    dropout=0.35,
    max_pool_scale=0.25,
    topk=3,
    global_weight=0.15,
    local_weight=0.85
):
    model = GlobalLocalFireClassifier(
        dropout=dropout,
        use_pretrained=use_pretrained,
        max_pool_scale=max_pool_scale,
        topk=topk,
        global_weight=global_weight,
        local_weight=local_weight
    )

    model.shared_classifier = set_finetune_mode(
        model.shared_classifier,
        finetune_mode=finetune_mode
    )

    return model


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "frozen_params": total_params - trainable_params,
        "trainable_ratio": trainable_params / max(total_params, 1)
    }

In [4]:
# =============
# bbox、tile、attention mask 与缓存工具
# =============

def safe_torch_load(path, map_location="cpu", weights_only=True):
    try:
        return torch.load(path, map_location=map_location, weights_only=weights_only)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_fire_bboxes(annotation_path):
    if annotation_path is None:
        return []

    annotation_path = str(annotation_path)

    if annotation_path == "" or annotation_path.lower() == "nan":
        return []

    p = Path(annotation_path)

    if not p.exists():
        return []

    try:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        return []

    bboxes = []

    for shape in data.get("shapes", []):
        label = str(shape.get("label", "")).lower().strip()

        if label != "fire":
            continue

        points = shape.get("points", [])

        if len(points) < 2:
            continue

        xs = [float(pt[0]) for pt in points]
        ys = [float(pt[1]) for pt in points]

        x1, x2 = min(xs), max(xs)
        y1, y2 = min(ys), max(ys)

        if x2 > x1 and y2 > y1:
            bboxes.append((x1, y1, x2, y2))

    return bboxes


def intersection_area(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    return max(0, ix2 - ix1) * max(0, iy2 - iy1)


def expand_box(box, image_w, image_h, ratio=0.20):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1

    ex = bw * ratio
    ey = bh * ratio

    return (
        max(0, x1 - ex),
        max(0, y1 - ey),
        min(image_w, x2 + ex),
        min(image_h, y2 + ey)
    )


def dilate_mask_np(mask, radius=1):
    if radius <= 0:
        return mask

    out = mask.copy()
    H, W = mask.shape

    ys, xs = np.where(mask > 0)

    for y, x in zip(ys, xs):
        y1 = max(0, y - radius)
        y2 = min(H, y + radius + 1)
        x1 = max(0, x - radius)
        x2 = min(W, x + radius + 1)
        out[y1:y2, x1:x2] = 1.0

    return out


def make_attention_mask_from_bboxes(
    bboxes,
    image_w,
    image_h,
    target_h,
    target_w,
    expand_ratio=0.20,
    dilate=1
):
    mask = np.zeros((target_h, target_w), dtype=np.float32)

    if len(bboxes) == 0:
        return torch.zeros(1, target_h, target_w, dtype=torch.float32), torch.tensor(False)

    for bbox in bboxes:
        x1, y1, x2, y2 = expand_box(
            bbox,
            image_w=image_w,
            image_h=image_h,
            ratio=expand_ratio
        )

        gx1 = int(np.floor(x1 / max(image_w, 1) * target_w))
        gx2 = int(np.ceil(x2 / max(image_w, 1) * target_w))
        gy1 = int(np.floor(y1 / max(image_h, 1) * target_h))
        gy2 = int(np.ceil(y2 / max(image_h, 1) * target_h))

        gx1 = max(0, min(target_w - 1, gx1))
        gx2 = max(1, min(target_w, gx2))
        gy1 = max(0, min(target_h - 1, gy1))
        gy2 = max(1, min(target_h, gy2))

        mask[gy1:gy2, gx1:gx2] = 1.0

    mask = dilate_mask_np(mask, radius=dilate)
    valid = bool(mask.sum() > 0)

    return torch.tensor(mask[None, :, :], dtype=torch.float32), torch.tensor(valid)


def make_attention_mask_for_crop(
    bboxes,
    crop_box,
    target_h,
    target_w,
    expand_ratio=0.20,
    dilate=1
):
    cx1, cy1, cx2, cy2 = crop_box
    crop_w = cx2 - cx1
    crop_h = cy2 - cy1

    local_boxes = []

    for bbox in bboxes:
        ix1 = max(cx1, bbox[0])
        iy1 = max(cy1, bbox[1])
        ix2 = min(cx2, bbox[2])
        iy2 = min(cy2, bbox[3])

        if ix2 > ix1 and iy2 > iy1:
            local_boxes.append((
                ix1 - cx1,
                iy1 - cy1,
                ix2 - cx1,
                iy2 - cy1
            ))

    return make_attention_mask_from_bboxes(
        bboxes=local_boxes,
        image_w=crop_w,
        image_h=crop_h,
        target_h=target_h,
        target_w=target_w,
        expand_ratio=expand_ratio,
        dilate=dilate
    )


def make_bright_attention_mask(image, target_h, target_w, percentile=90):
    gray = image.convert("L").resize((target_w, target_h), Image.BILINEAR)
    arr = np.asarray(gray, dtype=np.float32)
    thr = np.percentile(arr, percentile)

    mask = (arr >= thr).astype(np.float32)

    if mask.sum() <= 0:
        return torch.zeros(1, target_h, target_w, dtype=torch.float32)

    return torch.tensor(mask[None, :, :], dtype=torch.float32)


def crop_box_from_center(cx, cy, crop_size, image_w, image_h):
    crop_w = min(int(crop_size), int(image_w))
    crop_h = min(int(crop_size), int(image_h))

    x1 = int(round(cx - crop_w / 2))
    y1 = int(round(cy - crop_h / 2))

    x1 = max(0, min(x1, image_w - crop_w))
    y1 = max(0, min(y1, image_h - crop_h))

    return (x1, y1, x1 + crop_w, y1 + crop_h)


def random_crop_box(image_w, image_h, crop_size):
    crop_w = min(int(crop_size), int(image_w))
    crop_h = min(int(crop_size), int(image_h))

    x1 = 0 if image_w <= crop_w else np.random.randint(0, image_w - crop_w + 1)
    y1 = 0 if image_h <= crop_h else np.random.randint(0, image_h - crop_h + 1)

    return (x1, y1, x1 + crop_w, y1 + crop_h)


def sample_positive_crop_box(bboxes, image_w, image_h):
    bbox = bboxes[np.random.randint(0, len(bboxes))]
    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1

    base_size = int(np.random.choice(LOCAL_TILE_SIZES))
    crop_size = max(base_size, int(max(bw, bh) * 4))
    crop_size = min(crop_size, max(image_w, image_h))

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    jitter = crop_size * 0.15
    cx = cx + np.random.uniform(-jitter, jitter)
    cy = cy + np.random.uniform(-jitter, jitter)

    return crop_box_from_center(cx, cy, crop_size, image_w, image_h)


def sample_negative_crop_box(image_w, image_h, bboxes=None, max_try=80):
    if bboxes is None:
        bboxes = []

    for _ in range(max_try):
        crop_size = int(np.random.choice(LOCAL_TILE_SIZES))
        crop_box = random_crop_box(image_w, image_h, crop_size)

        if all(intersection_area(crop_box, bbox) <= 0 for bbox in bboxes):
            return crop_box

    crop_size = int(np.random.choice(LOCAL_TILE_SIZES))
    return random_crop_box(image_w, image_h, crop_size)


def crop_brightness(image, crop_box):
    crop = image.crop(crop_box).resize((64, 64))
    arr = np.asarray(crop.convert("L"), dtype=np.float32)
    return float(arr.mean() + 0.5 * arr.max())


def sample_bright_negative_crop_box(image, image_w, image_h, candidate_num=24):
    candidates = []

    for _ in range(candidate_num):
        crop_size = int(np.random.choice(LOCAL_TILE_SIZES))
        crop_box = random_crop_box(image_w, image_h, crop_size)
        score = crop_brightness(image, crop_box)
        candidates.append((score, crop_box))

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    if np.random.rand() < 0.8:
        return candidates[0][1]

    return candidates[np.random.randint(0, len(candidates))][1]


def make_eval_grid_boxes(image_w, image_h, tile_size=512, stride=256):
    tile_w = min(int(tile_size), int(image_w))
    tile_h = min(int(tile_size), int(image_h))

    if image_w <= tile_w:
        xs = [0]
    else:
        xs = list(range(0, image_w - tile_w + 1, stride))
        if xs[-1] != image_w - tile_w:
            xs.append(image_w - tile_w)

    if image_h <= tile_h:
        ys = [0]
    else:
        ys = list(range(0, image_h - tile_h + 1, stride))
        if ys[-1] != image_h - tile_h:
            ys.append(image_h - tile_h)

    return [(x, y, x + tile_w, y + tile_h) for y in ys for x in xs]


def safe_stem(name):
    return Path(str(name)).stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def pil_to_uint8_chw(image):
    image = image.convert("RGB")
    arr = np.asarray(image, dtype=np.uint8)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()


def resize_pil_to_uint8_chw(image, size):
    image = image.convert("RGB").resize((size, size), Image.BILINEAR)
    return pil_to_uint8_chw(image)


def crop_resize_to_uint8_chw(image, crop_box, size):
    crop = image.crop(crop_box).convert("RGB").resize((size, size), Image.BILINEAR)
    return pil_to_uint8_chw(crop)


def get_cache_path(cache_dir, dataset_tag, idx, filename):
    cache_dir = Path(cache_dir) / dataset_tag
    cache_dir.mkdir(parents=True, exist_ok=True)
    stem = safe_stem(filename)
    return cache_dir / f"{idx:06d}_{stem}.pt"


def resolve_image_path_from_row(row, image_dir):
    filename = str(row["filename"])

    if "path" in row.index and pd.notna(row.get("path", "")):
        image_path = Path(str(row["path"]))
        if image_path.exists():
            return image_path

    return Path(image_dir) / filename


def make_cached_train_tiles(image, label, bboxes, attn_h, attn_w):
    image_w, image_h = image.size

    tile_tensors = []
    tile_labels = []
    tile_attn_masks = []
    tile_attn_valid = []

    if label == 1 and len(bboxes) > 0:
        for _ in range(TRAIN_POS_TILE_POOL_SIZE):
            crop_box = sample_positive_crop_box(bboxes, image_w, image_h)
            tile_tensors.append(crop_resize_to_uint8_chw(image, crop_box, TILE_IMG_SIZE))
            tile_labels.append(1.0)

            mask, valid = make_attention_mask_for_crop(
                bboxes=bboxes,
                crop_box=crop_box,
                target_h=attn_h,
                target_w=attn_w,
                expand_ratio=ATTN_BOX_EXPAND_RATIO,
                dilate=ATTN_MASK_DILATE
            )
            tile_attn_masks.append(mask)
            tile_attn_valid.append(bool(valid.item()))

        for _ in range(TRAIN_NEG_TILE_POOL_SIZE):
            crop_box = sample_negative_crop_box(image_w, image_h, bboxes)
            tile_tensors.append(crop_resize_to_uint8_chw(image, crop_box, TILE_IMG_SIZE))
            tile_labels.append(0.0)
            tile_attn_masks.append(torch.zeros(1, attn_h, attn_w, dtype=torch.float32))
            tile_attn_valid.append(False)

    elif label == 0:
        for _ in range(TRAIN_NO_FIRE_TILE_POOL_SIZE):
            crop_box = sample_bright_negative_crop_box(
                image=image,
                image_w=image_w,
                image_h=image_h,
                candidate_num=CACHE_BRIGHT_CANDIDATE_NUM
            )
            tile_tensors.append(crop_resize_to_uint8_chw(image, crop_box, TILE_IMG_SIZE))
            tile_labels.append(0.0)
            tile_attn_masks.append(torch.zeros(1, attn_h, attn_w, dtype=torch.float32))
            tile_attn_valid.append(False)

    else:
        for _ in range(TRAIN_TILE_POOL_SIZE):
            crop_box = random_crop_box(image_w, image_h, int(np.random.choice(LOCAL_TILE_SIZES)))
            tile_tensors.append(crop_resize_to_uint8_chw(image, crop_box, TILE_IMG_SIZE))
            tile_labels.append(-1.0)
            tile_attn_masks.append(torch.zeros(1, attn_h, attn_w, dtype=torch.float32))
            tile_attn_valid.append(False)

    if len(tile_tensors) == 0:
        tile_tensors.append(resize_pil_to_uint8_chw(image, TILE_IMG_SIZE))
        tile_labels.append(-1.0)
        tile_attn_masks.append(torch.zeros(1, attn_h, attn_w, dtype=torch.float32))
        tile_attn_valid.append(False)

    return (
        torch.stack(tile_tensors, dim=0),
        torch.tensor(tile_labels, dtype=torch.float32),
        torch.stack(tile_attn_masks, dim=0),
        torch.tensor(tile_attn_valid, dtype=torch.bool)
    )


def make_cached_eval_tiles(image):
    image_w, image_h = image.size
    boxes = make_eval_grid_boxes(image_w, image_h, EVAL_TILE_SIZE, EVAL_STRIDE)

    if len(boxes) > N_EVAL_TILES:
        idxs = np.linspace(0, len(boxes) - 1, N_EVAL_TILES).astype(int)
        boxes = [boxes[i] for i in idxs]

    tile_tensors = [crop_resize_to_uint8_chw(image, crop_box, TILE_IMG_SIZE) for crop_box in boxes]
    tile_mask = [True] * len(tile_tensors)

    while len(tile_tensors) < N_EVAL_TILES:
        tile_tensors.append(resize_pil_to_uint8_chw(image, TILE_IMG_SIZE))
        tile_mask.append(False)

    tiles_uint8 = torch.stack(tile_tensors, dim=0)
    tile_mask = torch.tensor(tile_mask, dtype=torch.bool)
    tile_labels = torch.full((len(tile_tensors),), -1.0, dtype=torch.float32)

    return tiles_uint8, tile_labels, tile_mask

In [5]:
# =============
# Transform：在线模式 + 缓存模式
# =============

global_train_transform = transforms.Compose([
    transforms.Resize((GLOBAL_IMG_SIZE, GLOBAL_IMG_SIZE)),
    transforms.ColorJitter(
        brightness=0.35,
        contrast=0.30,
        saturation=0.20,
        hue=0.03
    ),
    transforms.RandomApply([transforms.RandomAutocontrast()], p=0.30),
    transforms.RandomApply([transforms.RandomEqualize()], p=0.20),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

tile_train_transform = transforms.Compose([
    transforms.Resize((TILE_IMG_SIZE, TILE_IMG_SIZE)),
    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.12,
        hue=0.015
    ),
    transforms.RandomApply([transforms.RandomAutocontrast()], p=0.20),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))
    ], p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

global_eval_transform = transforms.Compose([
    transforms.Resize((GLOBAL_IMG_SIZE, GLOBAL_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

tile_eval_transform = transforms.Compose([
    transforms.Resize((TILE_IMG_SIZE, TILE_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

cached_global_train_transform = transforms.Compose([
    # 这些增强需要 uint8，所以放在最前面
    transforms.RandomApply([
        transforms.RandomAutocontrast()
    ], p=0.30),

    transforms.RandomEqualize(p=0.20),

    # 从这里开始转 float32，值域变为 0~1
    transforms.ConvertImageDtype(torch.float32),

    transforms.ColorJitter(
        brightness=0.35,
        contrast=0.30,
        saturation=0.20,
        hue=0.03
    ),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


cached_tile_train_transform = transforms.Compose([
    # 这些增强需要 uint8，所以放在最前面
    transforms.RandomApply([
        transforms.RandomAutocontrast()
    ], p=0.20),

    # tile 分支不建议过强 equalize，先不加 RandomEqualize
    # 如果一定要加，也必须放在 ConvertImageDtype 之前：
    # transforms.RandomEqualize(p=0.10),

    transforms.ConvertImageDtype(torch.float32),

    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.12,
        hue=0.015
    ),

    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 0.8)
        )
    ], p=0.05),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


cached_global_eval_transform = transforms.Compose([
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


cached_tile_eval_transform = transforms.Compose([
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
# =============
# Dataset、DataLoader
# =============

class GlobalLocalFireDataset(Dataset):
    def __init__(
        self,
        csv_path,
        image_dir,
        global_transform,
        tile_transform,
        train=True,
        n_train_tiles=8,
        n_eval_tiles=16,
        cache_mode="none",
        cache_root=None,
        rebuild_cache=False,
        dataset_tag=None,
        load_disk_cache_to_memory=False
    ):
        self.csv_path = Path(csv_path)
        self.df = pd.read_csv(csv_path)
        self.image_dir = Path(image_dir)

        self.global_transform = global_transform
        self.tile_transform = tile_transform

        self.train = bool(train)
        self.n_train_tiles = int(n_train_tiles)
        self.n_eval_tiles = int(n_eval_tiles)

        self.cache_mode = str(cache_mode).lower()
        self.cache_root = Path(cache_root) if cache_root is not None else None
        self.rebuild_cache = bool(rebuild_cache)
        self.load_disk_cache_to_memory = bool(load_disk_cache_to_memory)

        if dataset_tag is None:
            dataset_tag = "train" if self.train else "eval"

        self.dataset_tag = str(dataset_tag)

        self.df["label"] = self.df["label"].astype(int)

        if "annotation_path" not in self.df.columns:
            self.df["annotation_path"] = ""

        if "has_fire_box" not in self.df.columns:
            self.df["has_fire_box"] = False

        if self.cache_mode not in ["none", "disk", "memory"]:
            raise ValueError(f"未知 CACHE_MODE: {self.cache_mode}")

        self.bbox_cache = {}
        self.image_path_cache = {}

        for i, row in self.df.iterrows():
            self.image_path_cache[i] = resolve_image_path_from_row(row, self.image_dir)
            ann_path = str(row.get("annotation_path", ""))

            if ann_path == "" or ann_path.lower() == "nan":
                self.bbox_cache[i] = []
            else:
                self.bbox_cache[i] = load_fire_bboxes(ann_path)

        self.cache_paths = {}

        if self.cache_mode in ["disk", "memory"]:
            if self.cache_root is None:
                raise ValueError("cache_mode 为 disk/memory 时，cache_root 不能为空。")

            self.cache_root.mkdir(parents=True, exist_ok=True)
            self._build_disk_cache_if_needed()

            self.cache_paths = {
                i: get_cache_path(self.cache_root, self.dataset_tag, i, self.df.iloc[i]["filename"])
                for i in range(len(self.df))
            }

        self.memory_cache = {}

        if self.cache_mode == "memory":
            print(f"\n正在将 {self.dataset_tag} 缓存加载到内存...")
            for i in tqdm(range(len(self.df)), desc=f"load memory cache {self.dataset_tag}"):
                self.memory_cache[i] = safe_torch_load(self.cache_paths[i], map_location="cpu", weights_only=True)
            print(f"{self.dataset_tag} 内存缓存完成: {len(self.memory_cache)} items")

        elif self.cache_mode == "disk" and self.load_disk_cache_to_memory:
            print(f"\n正在将 {self.dataset_tag} 磁盘缓存预读到内存...")
            for i in tqdm(range(len(self.df)), desc=f"preload disk cache {self.dataset_tag}"):
                self.memory_cache[i] = safe_torch_load(self.cache_paths[i], map_location="cpu", weights_only=True)
            print(f"{self.dataset_tag} 磁盘缓存预读完成: {len(self.memory_cache)} items")

    def __len__(self):
        return len(self.df)

    def _build_disk_cache_if_needed(self):
        cache_dir = Path(self.cache_root) / self.dataset_tag
        cache_dir.mkdir(parents=True, exist_ok=True)

        need_build = self.rebuild_cache

        if not need_build:
            expected_paths = [
                get_cache_path(self.cache_root, self.dataset_tag, i, self.df.iloc[i]["filename"])
                for i in range(len(self.df))
            ]
            missing = [p for p in expected_paths if not p.exists()]
            if len(missing) > 0:
                need_build = True
                print(f"\n{self.dataset_tag} 缓存缺失 {len(missing)} 个文件，将补建缓存。")

        if not need_build:
            print(f"\n使用已有 {self.dataset_tag} 缓存: {cache_dir}")
            return

        print(f"\n开始构建 {self.dataset_tag} 磁盘缓存: {cache_dir}")

        global_attn_h = GLOBAL_IMG_SIZE // 32
        global_attn_w = GLOBAL_IMG_SIZE // 32
        tile_attn_h = TILE_IMG_SIZE // 32
        tile_attn_w = TILE_IMG_SIZE // 32

        for i in tqdm(range(len(self.df)), desc=f"build cache {self.dataset_tag}"):
            row = self.df.iloc[i]
            label = int(row["label"])
            filename = str(row["filename"])
            image_path = self.image_path_cache[i]
            bboxes = self.bbox_cache.get(i, [])

            cache_path = get_cache_path(self.cache_root, self.dataset_tag, i, filename)
            if cache_path.exists() and not self.rebuild_cache:
                continue

            try:
                image = Image.open(image_path).convert("RGB")
            except Exception as e:
                raise RuntimeError(f"读取图片失败: {image_path}, error={repr(e)}")

            image_w, image_h = image.size
            global_uint8 = resize_pil_to_uint8_chw(image, GLOBAL_IMG_SIZE)

            global_attn_mask, global_attn_valid = make_attention_mask_from_bboxes(
                bboxes=bboxes if label == 1 else [],
                image_w=image_w,
                image_h=image_h,
                target_h=global_attn_h,
                target_w=global_attn_w,
                expand_ratio=ATTN_BOX_EXPAND_RATIO,
                dilate=ATTN_MASK_DILATE
            )

            global_bright_mask = make_bright_attention_mask(
                image=image,
                target_h=global_attn_h,
                target_w=global_attn_w,
                percentile=BRIGHT_PERCENTILE
            )

            if self.train:
                train_tiles_uint8, train_tile_labels, train_tile_attn_masks, train_tile_attn_valid = make_cached_train_tiles(
                    image=image,
                    label=label,
                    bboxes=bboxes,
                    attn_h=tile_attn_h,
                    attn_w=tile_attn_w
                )

                payload = {
                    "filename": filename,
                    "image_path": str(image_path),
                    "label": int(label),
                    "has_box": bool(label == 1 and len(bboxes) > 0),
                    "bboxes": bboxes,
                    "global_uint8": global_uint8,
                    "global_attn_mask": global_attn_mask,
                    "global_attn_valid": global_attn_valid,
                    "global_bright_mask": global_bright_mask,
                    "train_tiles_uint8": train_tiles_uint8,
                    "train_tile_labels": train_tile_labels,
                    "train_tile_attn_masks": train_tile_attn_masks,
                    "train_tile_attn_valid": train_tile_attn_valid,
                    "cache_version": CACHE_VERSION,
                    "global_img_size": GLOBAL_IMG_SIZE,
                    "tile_img_size": TILE_IMG_SIZE
                }

            else:
                eval_tiles_uint8, eval_tile_labels, eval_tile_mask = make_cached_eval_tiles(image)

                payload = {
                    "filename": filename,
                    "image_path": str(image_path),
                    "label": int(label),
                    "has_box": bool(label == 1 and len(bboxes) > 0),
                    "bboxes": bboxes,
                    "global_uint8": global_uint8,
                    "global_attn_mask": global_attn_mask,
                    "global_attn_valid": global_attn_valid,
                    "global_bright_mask": global_bright_mask,
                    "eval_tiles_uint8": eval_tiles_uint8,
                    "eval_tile_labels": eval_tile_labels,
                    "eval_tile_mask": eval_tile_mask,
                    "cache_version": CACHE_VERSION,
                    "global_img_size": GLOBAL_IMG_SIZE,
                    "tile_img_size": TILE_IMG_SIZE
                }

            torch.save(payload, cache_path)

        print(f"{self.dataset_tag} 磁盘缓存构建完成: {cache_dir}")

    def _load_cached_item(self, idx):
        if idx in self.memory_cache:
            return self.memory_cache[idx]

        return safe_torch_load(self.cache_paths[idx], map_location="cpu", weights_only=True)

    def _get_cached_sample(self, idx):
        item = self._load_cached_item(idx)

        label = int(item["label"])
        filename = str(item["filename"])
        has_box = bool(item.get("has_box", False))

        global_uint8 = item["global_uint8"]

        if self.train:
            global_image = cached_global_train_transform(global_uint8)

            pool_tiles = item["train_tiles_uint8"]
            pool_labels = item["train_tile_labels"]
            pool_attn_masks = item["train_tile_attn_masks"]
            pool_attn_valid = item["train_tile_attn_valid"]

            pool_n = pool_tiles.shape[0]

            if pool_n >= self.n_train_tiles:
                chosen = torch.randperm(pool_n)[:self.n_train_tiles]
            else:
                chosen = torch.randint(low=0, high=pool_n, size=(self.n_train_tiles,))

            selected_tiles = pool_tiles[chosen]
            selected_tile_labels = pool_labels[chosen]
            selected_tile_attn_masks = pool_attn_masks[chosen]
            selected_tile_attn_valid = pool_attn_valid[chosen]

            tile_tensors = [
                cached_tile_train_transform(selected_tiles[j])
                for j in range(selected_tiles.shape[0])
            ]

            tiles = torch.stack(tile_tensors, dim=0)
            tile_labels = selected_tile_labels.float()
            tile_mask = torch.ones(self.n_train_tiles, dtype=torch.bool)
            tile_attn_masks = selected_tile_attn_masks.float()
            tile_attn_valid = selected_tile_attn_valid.bool()

        else:
            global_image = cached_global_eval_transform(global_uint8)

            selected_tiles = item["eval_tiles_uint8"]
            tile_labels = item["eval_tile_labels"].float()
            tile_mask = item["eval_tile_mask"].bool()

            tile_tensors = [
                cached_tile_eval_transform(selected_tiles[j])
                for j in range(selected_tiles.shape[0])
            ]

            tiles = torch.stack(tile_tensors, dim=0)

            tile_attn_h = TILE_IMG_SIZE // 32
            tile_attn_w = TILE_IMG_SIZE // 32
            tile_attn_masks = torch.zeros(selected_tiles.shape[0], 1, tile_attn_h, tile_attn_w)
            tile_attn_valid = torch.zeros(selected_tiles.shape[0], dtype=torch.bool)

        return {
            "global_image": global_image,
            "tiles": tiles,
            "label": torch.tensor(label, dtype=torch.float32),
            "tile_labels": tile_labels,
            "tile_mask": tile_mask,
            "has_box": torch.tensor(has_box, dtype=torch.bool),
            "global_attn_mask": item["global_attn_mask"].float(),
            "global_attn_valid": item["global_attn_valid"].bool(),
            "global_bright_mask": item["global_bright_mask"].float(),
            "tile_attn_masks": tile_attn_masks.float(),
            "tile_attn_valid": tile_attn_valid.bool(),
            "filename": filename
        }

    def _get_online_sample(self, idx):
        raise RuntimeError("当前修改版建议使用 CACHE_MODE='disk' 或 'memory'。如需在线模式，请先扩展在线 attention mask 返回逻辑。")

    def __getitem__(self, idx):
        if self.cache_mode in ["disk", "memory"]:
            return self._get_cached_sample(idx)

        return self._get_online_sample(idx)


def make_loader(csv_path, train, shuffle, use_sample_weight=False):
    dataset_tag = "train_" + Path(csv_path).stem if train else "eval_" + Path(csv_path).stem

    dataset = GlobalLocalFireDataset(
        csv_path=csv_path,
        image_dir=IMAGE_DIR,
        global_transform=global_train_transform if train else global_eval_transform,
        tile_transform=tile_train_transform if train else tile_eval_transform,
        train=train,
        n_train_tiles=N_TRAIN_TILES,
        n_eval_tiles=N_EVAL_TILES,
        cache_mode=CACHE_MODE,
        cache_root=CACHE_ROOT,
        rebuild_cache=REBUILD_CACHE,
        dataset_tag=dataset_tag,
        load_disk_cache_to_memory=LOAD_DISK_CACHE_TO_MEMORY
    )

    df = pd.read_csv(csv_path)

    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available()
    }

    if NUM_WORKERS > 0:
        loader_kwargs.update({
            "persistent_workers": True,
            "prefetch_factor": 2
        })

    if use_sample_weight and "sample_weight" in df.columns:
        sample_weights = df["sample_weight"].astype(float).values

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        loader = DataLoader(dataset, sampler=sampler, shuffle=False, **loader_kwargs)

    else:
        loader = DataLoader(dataset, shuffle=shuffle, **loader_kwargs)

    return dataset, loader


val_dataset, val_loader = make_loader(
    csv_path=VAL_CSV,
    train=False,
    shuffle=False,
    use_sample_weight=False
)

print("VAL_CSV:", VAL_CSV)
print("Val dataset:", len(val_dataset))
print("Val distribution:")
print(val_dataset.df["label"].value_counts().sort_index())


使用已有 eval_val_fixed_real_only 缓存: E:\Programming\Python\DeepLearning\比赛\fire_splits\tile_cache\v2_attn_global512_tile384_pool24_eval16\eval_val_fixed_real_only
VAL_CSV: E:\Programming\Python\DeepLearning\比赛\fire_splits\val_fixed_real_only.csv
Val dataset: 100
Val distribution:
label
0    50
1    50
Name: count, dtype: int64


In [7]:
# =============
# loss、评估、保存工具函数
# =============

def get_checkpoint_path(ensemble_id, checkpoint_type):
    return MODEL_SAVE_DIR / f"{MODEL_PREFIX}_{ensemble_id:02d}_{checkpoint_type}.pth"


def make_optimizer(model, lr_head=LR_HEAD, lr_backbone=LR_BACKBONE, weight_decay=WEIGHT_DECAY):
    head_params = []
    backbone_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if "backbone" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    param_groups = []

    if len(head_params) > 0:
        param_groups.append({"params": head_params, "lr": lr_head})

    if len(backbone_params) > 0:
        param_groups.append({"params": backbone_params, "lr": lr_backbone})

    if len(param_groups) == 0:
        raise RuntimeError("没有任何可训练参数，请检查 FINETUNE_MODE。")

    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


def smooth_binary_labels(labels, smoothing=0.05):
    if smoothing <= 0:
        return labels
    return labels * (1.0 - 2.0 * smoothing) + smoothing


def tile_supervision_loss(tile_logits, tile_labels, tile_mask):
    known_mask = (tile_labels >= 0) & tile_mask

    if known_mask.sum().item() == 0:
        return tile_logits.sum() * 0.0

    return F.binary_cross_entropy_with_logits(
        tile_logits[known_mask],
        tile_labels[known_mask].float()
    )


def attention_inside_loss(attention, target_mask, valid_mask):
    """
    attention: [B,1,H,W]，softmax空间和为1
    target_mask: [B,1,H,W]，bbox mask
    valid_mask: [B]
    """
    valid_mask = valid_mask.bool()

    if valid_mask.sum().item() == 0:
        return attention.sum() * 0.0

    attn = attention[valid_mask]
    mask = target_mask[valid_mask].to(attn.device).float()

    mask_sum = mask.flatten(1).sum(dim=1, keepdim=True).clamp_min(1.0)
    mask = mask / mask_sum.view(-1, 1, 1, 1)

    inside = (attn * (mask > 0).float()).flatten(1).sum(dim=1).clamp_min(1e-6)

    return (-torch.log(inside)).mean()


def tile_attention_inside_loss(tile_attentions, tile_attn_masks, tile_attn_valid, tile_labels, tile_mask):
    valid = (
        tile_attn_valid.bool()
        & tile_mask.bool()
        & (tile_labels > 0.5)
    )

    if valid.sum().item() == 0:
        return tile_attentions.sum() * 0.0

    attn = tile_attentions[valid]
    mask = tile_attn_masks.to(attn.device).float()[valid]

    inside = (attn * (mask > 0).float()).flatten(1).sum(dim=1).clamp_min(1e-6)

    return (-torch.log(inside)).mean()


def no_fire_bright_attention_penalty(global_attention, global_bright_mask, labels):
    valid = labels.long() == 0

    if valid.sum().item() == 0:
        return global_attention.sum() * 0.0

    attn = global_attention[valid]
    mask = global_bright_mask.to(attn.device).float()[valid]

    penalty = (attn * (mask > 0).float()).flatten(1).sum(dim=1)

    return penalty.mean()


def compute_attention_metrics(global_attention, global_attn_mask, global_attn_valid, global_bright_mask, labels):
    metrics = {
        "fire_attn_inside": np.nan,
        "no_fire_attn_on_bright": np.nan
    }

    labels_int = labels.long()

    fire_valid = (labels_int == 1) & global_attn_valid.bool().to(labels.device)

    if fire_valid.sum().item() > 0:
        attn = global_attention[fire_valid]
        mask = global_attn_mask.to(attn.device).float()[fire_valid]
        inside = (attn * (mask > 0).float()).flatten(1).sum(dim=1)
        metrics["fire_attn_inside"] = float(inside.mean().detach().cpu().item())

    no_fire_valid = labels_int == 0

    if no_fire_valid.sum().item() > 0:
        attn = global_attention[no_fire_valid]
        mask = global_bright_mask.to(attn.device).float()[no_fire_valid]
        bright = (attn * (mask > 0).float()).flatten(1).sum(dim=1)
        metrics["no_fire_attn_on_bright"] = float(bright.mean().detach().cpu().item())

    return metrics


def ensure_history_keys(history):
    keys = [
        "epoch",
        "train_loss",
        "train_final_loss",
        "train_global_loss",
        "train_tile_loss",
        "train_global_attn_loss",
        "train_tile_attn_loss",
        "train_bright_attn_loss",
        "train_acc",
        "val_loss",
        "val_acc",
        "val_precision",
        "val_recall",
        "val_specificity",
        "val_f1",
        "val_balanced_acc",
        "val_tp",
        "val_tn",
        "val_fp",
        "val_fn",
        "val_mean_prob",
        "val_mean_global_prob",
        "val_mean_local_prob",
        "val_fire_attn_inside",
        "val_no_fire_attn_on_bright"
    ]

    n = len(history.get("epoch", []))

    for key in keys:
        if key not in history:
            history[key] = [np.nan] * n

    return history

In [8]:
# =============
# 训练与验证函数
# =============

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    show_step=True,
    step_log_interval=10,
    epoch=None,
    end_epoch=None,
    ensemble_id=None
):
    model.train()

    total_loss = 0.0
    total_final_loss = 0.0
    total_global_loss = 0.0
    total_tile_loss = 0.0
    total_global_attn_loss = 0.0
    total_tile_attn_loss = 0.0
    total_bright_attn_loss = 0.0
    total_correct = 0
    total_count = 0

    if show_step:
        desc_parts = []

        if epoch is not None and end_epoch is not None:
            desc_parts.append(f"Epoch {epoch}/{end_epoch}")

        if ensemble_id is not None:
            desc_parts.append(f"model {ensemble_id:02d}")

        desc = " | ".join(desc_parts) if desc_parts else "Training"

        step_iter = tqdm(
            enumerate(loader, start=1),
            total=len(loader),
            desc=desc,
            dynamic_ncols=True,
            leave=False
        )
    else:
        step_iter = enumerate(loader, start=1)

    for step, batch in step_iter:
        global_images = batch["global_image"].to(DEVICE, non_blocking=True)
        tiles = batch["tiles"].to(DEVICE, non_blocking=True)
        tile_labels = batch["tile_labels"].to(DEVICE, non_blocking=True)
        tile_mask = batch["tile_mask"].to(DEVICE, non_blocking=True)
        has_box = batch["has_box"].to(DEVICE, non_blocking=True)

        labels = batch["label"].float().to(DEVICE, non_blocking=True)
        labels_int = labels.long()

        global_attn_mask = batch["global_attn_mask"].to(DEVICE, non_blocking=True)
        global_attn_valid = batch["global_attn_valid"].to(DEVICE, non_blocking=True)
        global_bright_mask = batch["global_bright_mask"].to(DEVICE, non_blocking=True)
        tile_attn_masks = batch["tile_attn_masks"].to(DEVICE, non_blocking=True)
        tile_attn_valid = batch["tile_attn_valid"].to(DEVICE, non_blocking=True)

        labels_smooth = smooth_binary_labels(labels, smoothing=LABEL_SMOOTHING)

        optimizer.zero_grad()

        outputs = model(
            global_image=global_images,
            tiles=tiles,
            tile_mask=tile_mask
        )

        final_logits = outputs["logit"].squeeze(1)
        global_logits = outputs["global_logit"].squeeze(1)
        tile_logits = outputs["tile_logits"]
        global_attention = outputs["global_attention"]
        tile_attentions = outputs["tile_attentions"]

        use_global_only = (labels_int == 1) & (~has_box)

        image_logits_for_loss = torch.where(
            use_global_only,
            global_logits,
            final_logits
        )

        final_loss = criterion(image_logits_for_loss, labels_smooth)
        global_loss = criterion(global_logits, labels_smooth)

        tile_loss = tile_supervision_loss(
            tile_logits=tile_logits,
            tile_labels=tile_labels,
            tile_mask=tile_mask
        )

        global_attn_loss = attention_inside_loss(
            attention=global_attention,
            target_mask=global_attn_mask,
            valid_mask=((labels_int == 1) & global_attn_valid.bool())
        )

        tile_attn_loss = tile_attention_inside_loss(
            tile_attentions=tile_attentions,
            tile_attn_masks=tile_attn_masks,
            tile_attn_valid=tile_attn_valid,
            tile_labels=tile_labels,
            tile_mask=tile_mask
        )

        bright_attn_loss = no_fire_bright_attention_penalty(
            global_attention=global_attention,
            global_bright_mask=global_bright_mask,
            labels=labels
        )

        loss = (
            FINAL_LOSS_WEIGHT * final_loss
            + GLOBAL_LOSS_WEIGHT * global_loss
            + TILE_LOSS_WEIGHT * tile_loss
            + GLOBAL_ATTN_LOSS_WEIGHT * global_attn_loss
            + TILE_ATTN_LOSS_WEIGHT * tile_attn_loss
            + NO_FIRE_BRIGHT_ATTN_WEIGHT * bright_attn_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM
        )

        optimizer.step()

        with torch.no_grad():
            probs = torch.sigmoid(final_logits)
            preds = (probs >= THRESHOLD).long()

            total_correct += (preds == labels_int).sum().item()
            total_count += labels_int.numel()

            bs = global_images.size(0)

            total_loss += loss.item() * bs
            total_final_loss += final_loss.item() * bs
            total_global_loss += global_loss.item() * bs
            total_tile_loss += tile_loss.item() * bs
            total_global_attn_loss += global_attn_loss.item() * bs
            total_tile_attn_loss += tile_attn_loss.item() * bs
            total_bright_attn_loss += bright_attn_loss.item() * bs

        if show_step and (step % step_log_interval == 0 or step == len(loader)):
            current_loss = total_loss / max(total_count, 1)
            current_acc = total_correct / max(total_count, 1)

            step_iter.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{current_loss:.4f}",
                "acc": f"{current_acc:.4f}",
                "attn": f"{(total_global_attn_loss / max(total_count, 1)):.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
            })

    return {
        "loss": total_loss / max(total_count, 1),
        "final_loss": total_final_loss / max(total_count, 1),
        "global_loss": total_global_loss / max(total_count, 1),
        "tile_loss": total_tile_loss / max(total_count, 1),
        "global_attn_loss": total_global_attn_loss / max(total_count, 1),
        "tile_attn_loss": total_tile_attn_loss / max(total_count, 1),
        "bright_attn_loss": total_bright_attn_loss / max(total_count, 1),
        "acc": total_correct / max(total_count, 1)
    }


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    threshold=0.5,
    show_step=True,
    desc="Evaluating"
):
    model.eval()

    total_loss = 0.0
    total_num = 0

    all_probs = []
    all_global_probs = []
    all_local_probs = []
    all_labels = []
    all_fire_inside = []
    all_no_fire_bright = []

    running_tp = 0
    running_tn = 0
    running_fp = 0
    running_fn = 0

    if show_step:
        step_iter = tqdm(
            enumerate(loader, start=1),
            total=len(loader),
            desc=desc,
            dynamic_ncols=True,
            leave=False
        )
    else:
        step_iter = enumerate(loader, start=1)

    for step, batch in step_iter:
        global_images = batch["global_image"].to(DEVICE, non_blocking=True)
        tiles = batch["tiles"].to(DEVICE, non_blocking=True)
        tile_mask = batch["tile_mask"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True).float()

        global_attn_mask = batch["global_attn_mask"].to(DEVICE, non_blocking=True)
        global_attn_valid = batch["global_attn_valid"].to(DEVICE, non_blocking=True)
        global_bright_mask = batch["global_bright_mask"].to(DEVICE, non_blocking=True)

        outputs = model(
            global_image=global_images,
            tiles=tiles,
            tile_mask=tile_mask
        )

        logits = outputs["logit"].squeeze(1)
        global_logits = outputs["global_logit"].squeeze(1)
        local_logits = outputs["local_logit"].squeeze(1)
        global_attention = outputs["global_attention"]

        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)
        global_probs = torch.sigmoid(global_logits)
        local_probs = torch.sigmoid(local_logits)

        batch_size = global_images.size(0)

        total_loss += loss.item() * batch_size
        total_num += batch_size

        all_probs.append(probs.detach().cpu())
        all_global_probs.append(global_probs.detach().cpu())
        all_local_probs.append(local_probs.detach().cpu())
        all_labels.append(labels.detach().cpu())

        attn_metrics = compute_attention_metrics(
            global_attention=global_attention,
            global_attn_mask=global_attn_mask,
            global_attn_valid=global_attn_valid,
            global_bright_mask=global_bright_mask,
            labels=labels
        )

        if not pd.isna(attn_metrics["fire_attn_inside"]):
            all_fire_inside.append(attn_metrics["fire_attn_inside"])

        if not pd.isna(attn_metrics["no_fire_attn_on_bright"]):
            all_no_fire_bright.append(attn_metrics["no_fire_attn_on_bright"])

        preds = (probs >= threshold).long()
        labels_int = labels.long()

        running_tp += ((preds == 1) & (labels_int == 1)).sum().item()
        running_tn += ((preds == 0) & (labels_int == 0)).sum().item()
        running_fp += ((preds == 1) & (labels_int == 0)).sum().item()
        running_fn += ((preds == 0) & (labels_int == 1)).sum().item()

        if show_step:
            running_acc = (running_tp + running_tn) / max(running_tp + running_tn + running_fp + running_fn, 1)
            running_recall = running_tp / max(running_tp + running_fn, 1)
            running_specificity = running_tn / max(running_tn + running_fp, 1)
            running_balanced_acc = (running_recall + running_specificity) / 2
            running_loss = total_loss / max(total_num, 1)

            step_iter.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{running_loss:.4f}",
                "acc": f"{running_acc:.4f}",
                "recall": f"{running_recall:.4f}",
                "spec": f"{running_specificity:.4f}",
                "bal_acc": f"{running_balanced_acc:.4f}"
            })

    all_probs = torch.cat(all_probs)
    all_global_probs = torch.cat(all_global_probs)
    all_local_probs = torch.cat(all_local_probs)
    all_labels = torch.cat(all_labels)

    preds = (all_probs >= threshold).long()
    labels_int = all_labels.long()

    tp = ((preds == 1) & (labels_int == 1)).sum().item()
    tn = ((preds == 0) & (labels_int == 0)).sum().item()
    fp = ((preds == 1) & (labels_int == 0)).sum().item()
    fn = ((preds == 0) & (labels_int == 1)).sum().item()

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    balanced_acc = (recall + specificity) / 2

    return {
        "loss": total_loss / max(total_num, 1),
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "balanced_acc": balanced_acc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "mean_prob": float(all_probs.mean().item()),
        "mean_global_prob": float(all_global_probs.mean().item()),
        "mean_local_prob": float(all_local_probs.mean().item()),
        "fire_attn_inside": float(np.mean(all_fire_inside)) if len(all_fire_inside) > 0 else np.nan,
        "no_fire_attn_on_bright": float(np.mean(all_no_fire_bright)) if len(all_no_fire_bright) > 0 else np.nan
    }

In [9]:
# =============
# 曲线、checkpoint 与训练状态
# =============

def save_loss_curve(history, save_path):
    fig, axes = plt.subplots(1, 5, figsize=(30, 5))

    for ensemble_id in sorted(history.keys()):
        h = ensure_history_keys(history[ensemble_id])

        axes[0].plot(h["epoch"], h["train_loss"], marker="o", label=f"model {ensemble_id:02d}")
        axes[1].plot(h["epoch"], h["val_loss"], marker="o", label=f"model {ensemble_id:02d}")
        axes[2].plot(h["epoch"], h["train_tile_loss"], marker="o", label=f"model {ensemble_id:02d}")
        axes[3].plot(h["epoch"], h["train_global_attn_loss"], marker="o", label=f"model {ensemble_id:02d}")
        axes[4].plot(h["epoch"], h["train_tile_attn_loss"], marker="o", label=f"model {ensemble_id:02d}")

    titles = ["Train Loss", "Validation Loss", "Train Tile Loss", "Global Attn Loss", "Tile Attn Loss"]

    for ax, title in zip(axes, titles):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True)

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_metric_curve(history, save_path):
    fig, axes = plt.subplots(1, 7, figsize=(36, 5))

    for ensemble_id in sorted(history.keys()):
        h = ensure_history_keys(history[ensemble_id])

        axes[0].plot(h["epoch"], h["val_acc"], marker="o", label=f"model {ensemble_id:02d}")
        axes[1].plot(h["epoch"], h["val_recall"], marker="o", label=f"model {ensemble_id:02d}")
        axes[2].plot(h["epoch"], h["val_specificity"], marker="o", label=f"model {ensemble_id:02d}")
        axes[3].plot(h["epoch"], h["val_f1"], marker="o", label=f"model {ensemble_id:02d}")
        axes[4].plot(h["epoch"], h["val_balanced_acc"], marker="o", label=f"model {ensemble_id:02d}")
        axes[5].plot(h["epoch"], h["val_mean_prob"], marker="o", label=f"final {ensemble_id:02d}")
        axes[5].plot(h["epoch"], h["val_mean_global_prob"], marker="x", label=f"global {ensemble_id:02d}")
        axes[5].plot(h["epoch"], h["val_mean_local_prob"], marker="s", label=f"local {ensemble_id:02d}")
        axes[6].plot(h["epoch"], h["val_fire_attn_inside"], marker="o", label="fire inside")
        axes[6].plot(h["epoch"], h["val_no_fire_attn_on_bright"], marker="x", label="no_fire bright")

    titles = [
        "Validation Accuracy",
        "Validation Recall",
        "Validation Specificity",
        "Validation F1",
        "Validation Balanced Acc",
        "Mean Prob",
        "Attention Metrics"
    ]

    for i, (ax, title) in enumerate(zip(axes, titles)):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Score")
        if i < 5:
            ax.set_ylim(0, 1.05)
        ax.legend()
        ax.grid(True)

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def rebuild_all_history_rows(history):
    rows = []

    for ensemble_id in sorted(history.keys()):
        h = ensure_history_keys(history[ensemble_id])

        for i in range(len(h["epoch"])):
            rows.append({
                "ensemble_id": ensemble_id,
                **{key: h[key][i] for key in h.keys()}
            })

    return rows


def save_checkpoint(
    ensemble_id,
    epoch,
    model,
    optimizer,
    scheduler,
    history,
    best_score,
    checkpoint_type,
    extra=None
):
    payload = {
        "ensemble_id": ensemble_id,
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "history": history,
        "best_score": float(best_score),
        "best_score_name": "balanced_acc",
        "threshold": float(THRESHOLD),
        "model_name": "global_local_convnext_tiny_attention_supervised",
        "model_prefix": MODEL_PREFIX,
        "global_img_size": int(GLOBAL_IMG_SIZE),
        "tile_img_size": int(TILE_IMG_SIZE),
        "n_train_tiles": int(N_TRAIN_TILES),
        "n_eval_tiles": int(N_EVAL_TILES),
        "eval_tile_size": int(EVAL_TILE_SIZE),
        "eval_stride": int(EVAL_STRIDE),
        "local_tile_sizes": list(map(int, LOCAL_TILE_SIZES)),
        "finetune_mode": FINETUNE_MODE,
        "dropout": float(DROPOUT),
        "max_pool_scale": float(MAX_POOL_SCALE),
        "local_topk": int(LOCAL_TOPK),
        "global_weight": float(GLOBAL_WEIGHT),
        "local_weight": float(LOCAL_WEIGHT),
        "attention_loss": {
            "global_attn_loss_weight": float(GLOBAL_ATTN_LOSS_WEIGHT),
            "tile_attn_loss_weight": float(TILE_ATTN_LOSS_WEIGHT),
            "no_fire_bright_attn_weight": float(NO_FIRE_BRIGHT_ATTN_WEIGHT),
            "attn_box_expand_ratio": float(ATTN_BOX_EXPAND_RATIO),
            "attn_mask_dilate": int(ATTN_MASK_DILATE)
        },
        "lr_head": float(LR_HEAD),
        "lr_backbone": float(LR_BACKBONE),
        "weight_decay": float(WEIGHT_DECAY),
        "pos_weight": float(POS_WEIGHT),
        "label_smoothing": float(LABEL_SMOOTHING),
        "batch_size": int(BATCH_SIZE),
        "cache_version": CACHE_VERSION
    }

    if extra is not None:
        payload.update(extra)

    torch.save(payload, get_checkpoint_path(ensemble_id, checkpoint_type))


def safe_load_optimizer_state(optimizer, checkpoint, ensemble_id):
    if "optimizer_state_dict" not in checkpoint:
        print(f"model {ensemble_id:02d} | checkpoint 中没有 optimizer_state_dict，优化器重新初始化。")
        return

    try:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        print(f"model {ensemble_id:02d} | optimizer 状态已恢复。")
    except Exception as e:
        print(f"model {ensemble_id:02d} | optimizer 状态恢复失败，优化器重新初始化。")
        print("原因:", repr(e))


def safe_load_scheduler_state(scheduler, checkpoint, ensemble_id):
    if scheduler is None:
        return

    if "scheduler_state_dict" not in checkpoint:
        print(f"model {ensemble_id:02d} | checkpoint 中没有 scheduler_state_dict，scheduler 重新初始化。")
        return

    try:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        print(f"model {ensemble_id:02d} | scheduler 状态已恢复。")
    except Exception as e:
        print(f"model {ensemble_id:02d} | scheduler 状态恢复失败，scheduler 重新初始化。")
        print("原因:", repr(e))


def initialize_training_state(lr_head=LR_HEAD, lr_backbone=LR_BACKBONE):
    models_dict = {}
    optimizers_dict = {}
    schedulers_dict = {}
    train_loaders_dict = {}
    criteria_dict = {}
    history = {}
    best_score = {}
    best_val_loss = {}

    for ensemble_id in range(N_ENSEMBLE):
        train_csv = find_train_csv(ensemble_id)

        train_dataset, train_loader = make_loader(
            csv_path=train_csv,
            train=True,
            shuffle=True,
            use_sample_weight=True
        )

        model = build_global_local_fire_model(
            finetune_mode=FINETUNE_MODE,
            use_pretrained=USE_PRETRAINED,
            dropout=DROPOUT,
            max_pool_scale=MAX_POOL_SCALE,
            topk=LOCAL_TOPK,
            global_weight=GLOBAL_WEIGHT,
            local_weight=LOCAL_WEIGHT
        ).to(DEVICE)

        param_info = count_parameters(model)

        optimizer = make_optimizer(
            model=model,
            lr_head=lr_head,
            lr_backbone=lr_backbone,
            weight_decay=WEIGHT_DECAY
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=5
        )

        pos_weight_tensor = torch.tensor(
            [POS_WEIGHT],
            dtype=torch.float32,
            device=DEVICE
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

        models_dict[ensemble_id] = model
        optimizers_dict[ensemble_id] = optimizer
        schedulers_dict[ensemble_id] = scheduler
        train_loaders_dict[ensemble_id] = train_loader
        criteria_dict[ensemble_id] = criterion

        history[ensemble_id] = ensure_history_keys({})
        best_score[ensemble_id] = -float("inf")
        best_val_loss[ensemble_id] = float("inf")

        train_df = train_dataset.df.copy()
        train_fire = train_df[train_df["label"] == 1]

        if "has_fire_box" in train_fire.columns:
            fire_has_box = int((train_fire["has_fire_box"] == True).sum())
            fire_no_box = int((train_fire["has_fire_box"] == False).sum())
        else:
            fire_has_box = 0
            fire_no_box = len(train_fire)

        print(
            f"model {ensemble_id:02d} | "
            f"train_csv={train_csv} | "
            f"train={len(train_dataset)} | "
            f"val={len(val_dataset)} | "
            f"fire_has_box={fire_has_box} | "
            f"fire_no_box={fire_no_box} | "
            f"trainable={param_info['trainable_params']} | "
            f"trainable_ratio={param_info['trainable_ratio']:.6f}"
        )

    return {
        "models": models_dict,
        "optimizers": optimizers_dict,
        "schedulers": schedulers_dict,
        "train_loaders": train_loaders_dict,
        "criteria": criteria_dict,
        "history": history,
        "best_score": best_score,
        "best_val_loss": best_val_loss
    }


def find_train_csv(ensemble_id):
    hard_csv = SOURCE_ENSEMBLE_DIR / f"ensemble_train_{ensemble_id:02d}_hard_weighted.csv"
    normal_csv = SOURCE_ENSEMBLE_DIR / f"ensemble_train_{ensemble_id:02d}.csv"

    if hard_csv.exists():
        return hard_csv

    if normal_csv.exists():
        return normal_csv

    raise FileNotFoundError(f"未找到训练 CSV：{hard_csv} 或 {normal_csv}")


def load_checkpoints_if_needed(
    state,
    resume_training=False,
    resume_from="latest",
    reset_optimizer=False,
    reset_scheduler=False
):
    loaded_epochs = []

    if not resume_training:
        print("\n从 ImageNet 预训练 ConvNeXt-Tiny 重新 fine-tune。")
        return 1

    print("\n准备加载 checkpoint")
    print("resume_from:", resume_from)
    print("reset_optimizer:", reset_optimizer)
    print("reset_scheduler:", reset_scheduler)

    for ensemble_id in range(N_ENSEMBLE):
        ckpt_path = get_checkpoint_path(ensemble_id, resume_from)

        if not ckpt_path.exists():
            print(f"model {ensemble_id:02d} | 未找到 checkpoint: {ckpt_path}")
            print("该模型从当前初始化状态开始训练。")
            loaded_epochs.append(0)
            continue

        checkpoint = safe_torch_load(ckpt_path, map_location=DEVICE, weights_only=False)

        model = state["models"][ensemble_id]
        optimizer = state["optimizers"][ensemble_id]
        scheduler = state["schedulers"][ensemble_id]

        model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        print(f"model {ensemble_id:02d} | 已加载模型权重: {ckpt_path}")

        if not reset_optimizer:
            safe_load_optimizer_state(optimizer, checkpoint, ensemble_id)

        if not reset_scheduler:
            safe_load_scheduler_state(scheduler, checkpoint, ensemble_id)

        loaded_epoch = int(checkpoint.get("epoch", 0))
        loaded_epochs.append(loaded_epoch)

        if "history" in checkpoint:
            state["history"][ensemble_id] = ensure_history_keys(checkpoint["history"])

        if "best_score" in checkpoint:
            state["best_score"][ensemble_id] = float(checkpoint["best_score"])
        else:
            h = state["history"][ensemble_id]
            vals = [v for v in h.get("val_balanced_acc", []) if not pd.isna(v)]
            state["best_score"][ensemble_id] = max(vals) if vals else -float("inf")

        if "val_loss" in checkpoint:
            state["best_val_loss"][ensemble_id] = float(checkpoint["val_loss"])

        print(
            f"model {ensemble_id:02d} | "
            f"loaded_epoch={loaded_epoch} | "
            f"best_balanced_acc={state['best_score'][ensemble_id]:.6f}"
        )

    return max(loaded_epochs) + 1

In [10]:
# =============
# 主训练函数
# =============

def run_training(
    resume_training=False,
    resume_from="latest",
    end_epoch=EPOCHS,
    reset_optimizer=False,
    reset_scheduler=False,
    lr_head=LR_HEAD,
    lr_backbone=LR_BACKBONE,
    run_name="train",
    show_step=True,
    step_log_interval=10
):
    state = initialize_training_state(lr_head=lr_head, lr_backbone=lr_backbone)

    start_epoch = load_checkpoints_if_needed(
        state=state,
        resume_training=resume_training,
        resume_from=resume_from,
        reset_optimizer=reset_optimizer,
        reset_scheduler=reset_scheduler
    )

    models_dict = state["models"]
    optimizers_dict = state["optimizers"]
    schedulers_dict = state["schedulers"]
    train_loaders_dict = state["train_loaders"]
    criteria_dict = state["criteria"]
    history = state["history"]
    best_score = state["best_score"]
    best_val_loss = state["best_val_loss"]

    if start_epoch > end_epoch:
        print(f"\n当前 checkpoint 已经训练到 epoch {start_epoch - 1}，end_epoch={end_epoch}，不会继续训练。")
        return state

    print(f"\n开始训练：epoch {start_epoch} -> {end_epoch}")
    print("run_name:", run_name)

    start_time = time.time()

    for epoch in range(start_epoch, end_epoch + 1):
        print(f"\nEpoch {epoch}/{end_epoch}")

        for ensemble_id in range(N_ENSEMBLE):
            model = models_dict[ensemble_id]
            optimizer = optimizers_dict[ensemble_id]
            scheduler = schedulers_dict[ensemble_id]
            train_loader = train_loaders_dict[ensemble_id]
            criterion = criteria_dict[ensemble_id]

            print(f"\n开始训练 model {ensemble_id:02d}")

            train_metrics = train_one_epoch(
                model=model,
                loader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                show_step=show_step,
                step_log_interval=step_log_interval,
                epoch=epoch,
                end_epoch=end_epoch,
                ensemble_id=ensemble_id
            )

            val_metrics = evaluate(
                model=model,
                loader=val_loader,
                criterion=criterion,
                threshold=THRESHOLD,
                show_step=True,
                desc=f"Val | model {ensemble_id:02d} | epoch {epoch}"
            )

            val_loss = val_metrics["loss"]
            scheduler.step(val_loss)

            history[ensemble_id] = ensure_history_keys(history[ensemble_id])
            h = history[ensemble_id]

            h["epoch"].append(epoch)

            h["train_loss"].append(train_metrics["loss"])
            h["train_final_loss"].append(train_metrics["final_loss"])
            h["train_global_loss"].append(train_metrics["global_loss"])
            h["train_tile_loss"].append(train_metrics["tile_loss"])
            h["train_global_attn_loss"].append(train_metrics["global_attn_loss"])
            h["train_tile_attn_loss"].append(train_metrics["tile_attn_loss"])
            h["train_bright_attn_loss"].append(train_metrics["bright_attn_loss"])
            h["train_acc"].append(train_metrics["acc"])

            h["val_loss"].append(val_loss)
            h["val_acc"].append(val_metrics["acc"])
            h["val_precision"].append(val_metrics["precision"])
            h["val_recall"].append(val_metrics["recall"])
            h["val_specificity"].append(val_metrics["specificity"])
            h["val_f1"].append(val_metrics["f1"])
            h["val_balanced_acc"].append(val_metrics["balanced_acc"])
            h["val_tp"].append(val_metrics["tp"])
            h["val_tn"].append(val_metrics["tn"])
            h["val_fp"].append(val_metrics["fp"])
            h["val_fn"].append(val_metrics["fn"])
            h["val_mean_prob"].append(val_metrics["mean_prob"])
            h["val_mean_global_prob"].append(val_metrics["mean_global_prob"])
            h["val_mean_local_prob"].append(val_metrics["mean_local_prob"])
            h["val_fire_attn_inside"].append(val_metrics["fire_attn_inside"])
            h["val_no_fire_attn_on_bright"].append(val_metrics["no_fire_attn_on_bright"])

            current_score = val_metrics["balanced_acc"]
            current_loss = val_loss

            improved_balanced = current_score > best_score[ensemble_id] + 1e-4
            tied_but_lower_loss = (
                abs(current_score - best_score[ensemble_id]) <= 1e-4
                and current_loss < best_val_loss[ensemble_id]
            )

            improved = improved_balanced or tied_but_lower_loss

            if improved:
                best_score[ensemble_id] = current_score
                best_val_loss[ensemble_id] = current_loss

                save_checkpoint(
                    ensemble_id=ensemble_id,
                    epoch=epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    history=history[ensemble_id],
                    best_score=best_score[ensemble_id],
                    checkpoint_type="best",
                    extra={
                        "run_name": run_name,
                        "val_loss": val_loss,
                        "val_balanced_acc": val_metrics["balanced_acc"]
                    }
                )

            if current_loss < best_val_loss[ensemble_id] + 1e-8:
                save_checkpoint(
                    ensemble_id=ensemble_id,
                    epoch=epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    history=history[ensemble_id],
                    best_score=current_score,
                    checkpoint_type="best_val_loss",
                    extra={
                        "run_name": run_name,
                        "val_loss": val_loss,
                        "val_balanced_acc": val_metrics["balanced_acc"]
                    }
                )

            save_checkpoint(
                ensemble_id=ensemble_id,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                history=history[ensemble_id],
                best_score=best_score[ensemble_id],
                checkpoint_type="latest",
                extra={"run_name": run_name}
            )

            print(
                f"model {ensemble_id:02d} | "
                f"train_loss={train_metrics['loss']:.4f} | "
                f"tile_loss={train_metrics['tile_loss']:.4f} | "
                f"g_attn={train_metrics['global_attn_loss']:.4f} | "
                f"t_attn={train_metrics['tile_attn_loss']:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"acc={val_metrics['acc']:.4f} | "
                f"recall={val_metrics['recall']:.4f} | "
                f"specificity={val_metrics['specificity']:.4f} | "
                f"balanced_acc={val_metrics['balanced_acc']:.4f} | "
                f"fire_attn_inside={val_metrics['fire_attn_inside']:.4f} | "
                f"no_fire_bright_attn={val_metrics['no_fire_attn_on_bright']:.4f} | "
                f"best={'yes' if improved else 'no'}"
            )

        save_loss_curve(history=history, save_path=TRAIN_RESULT_DIR / "ensemble_loss_curves_latest.png")
        save_metric_curve(history=history, save_path=TRAIN_RESULT_DIR / "ensemble_metric_curves_latest.png")

        pd.DataFrame(rebuild_all_history_rows(history)).to_csv(
            TRAIN_RESULT_DIR / "ensemble_training_history_latest.csv",
            index=False,
            encoding="utf-8-sig"
        )

    for ensemble_id in range(N_ENSEMBLE):
        save_checkpoint(
            ensemble_id=ensemble_id,
            epoch=end_epoch,
            model=models_dict[ensemble_id],
            optimizer=optimizers_dict[ensemble_id],
            scheduler=schedulers_dict[ensemble_id],
            history=history[ensemble_id],
            best_score=best_score[ensemble_id],
            checkpoint_type="final",
            extra={"run_name": run_name}
        )

        pd.DataFrame(history[ensemble_id]).to_csv(
            TRAIN_RESULT_DIR / f"{MODEL_PREFIX}_{ensemble_id:02d}_history.csv",
            index=False,
            encoding="utf-8-sig"
        )

    save_loss_curve(history=history, save_path=TRAIN_RESULT_DIR / "ensemble_loss_curves_final.png")
    save_metric_curve(history=history, save_path=TRAIN_RESULT_DIR / "ensemble_metric_curves_final.png")

    best_summary = pd.DataFrame([
        {
            "ensemble_id": ensemble_id,
            "best_balanced_acc": best_score[ensemble_id],
            "best_val_loss": best_val_loss[ensemble_id],
            "latest_model_path": str(get_checkpoint_path(ensemble_id, "latest")),
            "best_model_path": str(get_checkpoint_path(ensemble_id, "best")),
            "best_val_loss_model_path": str(get_checkpoint_path(ensemble_id, "best_val_loss")),
            "final_model_path": str(get_checkpoint_path(ensemble_id, "final"))
        }
        for ensemble_id in range(N_ENSEMBLE)
    ])

    best_summary.to_csv(TRAIN_RESULT_DIR / "ensemble_best_summary.csv", index=False, encoding="utf-8-sig")

    elapsed = time.time() - start_time

    print("\n训练完成")
    print("模型保存目录:", MODEL_SAVE_DIR)
    print("训练记录保存目录:", TRAIN_RESULT_DIR)
    print(f"本次运行耗时: {elapsed / 60:.2f} 分钟")
    print("\n每个模型最佳 balanced_acc:")
    print(best_summary)

    return state

In [ ]:
# =============
# 初次训练
# =============
# 第一次使用 attention mask / tile cache 时：
# 1. REBUILD_CACHE = True
# 2. 跑完缓存生成和初训后，再把 REBUILD_CACHE 改为 False
#
# 注意：不要和继续训练 cell 一起 Run All。

state_initial = run_training(
    resume_training=False,
    resume_from="latest",
    end_epoch=EPOCHS,
    reset_optimizer=False,
    reset_scheduler=False,
    lr_head=LR_HEAD,
    lr_backbone=LR_BACKBONE,
    run_name="initial_global_local_attention_supervised",
    show_step=True,
    step_log_interval=10
)


使用已有 train_ensemble_train_00 缓存: E:\Programming\Python\DeepLearning\比赛\fire_splits\tile_cache\v2_attn_global512_tile384_pool24_eval16\train_ensemble_train_00
model 00 | train_csv=E:\Programming\Python\DeepLearning\比赛\fire_splits\ensemble_splits\ensemble_train_00.csv | train=1000 | val=100 | fire_has_box=499 | fire_no_box=1 | trainable=6212098 | trainable_ratio=0.212252

从 ImageNet 预训练 ConvNeXt-Tiny 重新 fine-tune。

开始训练：epoch 1 -> 15
run_name: initial_global_local_attention_supervised

Epoch 1/15

开始训练 model 00


Epoch 1/15 | model 00:   0%|          | 0/250 [00:00<?, ?it/s]

Val | model 00 | epoch 1:   0%|          | 0/25 [00:00<?, ?it/s]

model 00 | train_loss=1.4593 | tile_loss=0.4936 | g_attn=1.5969 | t_attn=0.8952 | val_loss=0.6792 | acc=0.6300 | recall=0.8000 | specificity=0.4600 | balanced_acc=0.6300 | fire_attn_inside=0.3474 | no_fire_bright_attn=0.1055 | best=yes

Epoch 2/15

开始训练 model 00


Epoch 2/15 | model 00:   0%|          | 0/250 [00:00<?, ?it/s]

...

model 00 | train_loss=0.6839 | tile_loss=0.2107 | g_attn=1.0505 | t_attn=0.4035 | val_loss=0.5475 | acc=0.7600 | recall=0.8400 | specificity=0.6800 | balanced_acc=0.7600 | fire_attn_inside=0.5422 | no_fire_bright_attn=0.1252 | best=no

Epoch 6/15

开始训练 model 00


Epoch 6/15 | model 00:   0%|          | 0/250 [00:00<?, ?it/s]

Val | model 00 | epoch 6:   0%|          | 0/25 [00:00<?, ?it/s]

model 00 | train_loss=0.6441 | tile_loss=0.2033 | g_attn=0.9902 | t_attn=0.3519 | val_loss=0.5332 | acc=0.7900 | recall=0.8600 | specificity=0.7200 | balanced_acc=0.7900 | fire_attn_inside=0.5720 | no_fire_bright_attn=0.1307 | best=yes

Epoch 7/15

开始训练 model 00


Epoch 7/15 | model 00:   0%|          | 0/250 [00:00<?, ?it/s]

In [11]:
# =============
# 检查当前 train / val / test 中 fire bbox 是否有效
# =============

def check_bbox_status(csv_path, name):
    df = pd.read_csv(csv_path)

    print("\n" + "=" * 80)
    print(name)
    print(csv_path)
    print("total:", len(df))
    print("label distribution:")
    print(df["label"].value_counts().sort_index())

    fire_df = df[df["label"] == 1].copy()
    print("\nfire total:", len(fire_df))

    if "annotation_path" not in fire_df.columns:
        print("没有 annotation_path 列")
        return fire_df

    fire_df["annotation_path_str"] = fire_df["annotation_path"].astype(str)
    fire_df["ann_not_empty"] = (
        (fire_df["annotation_path_str"] != "")
        & (fire_df["annotation_path_str"].str.lower() != "nan")
        & (fire_df["annotation_path_str"].str.lower() != "none")
    )

    fire_df["ann_exists"] = fire_df["annotation_path_str"].apply(
        lambda x: Path(x).exists()
        if x not in ["", "nan", "None", "none"]
        else False
    )

    print("fire annotation_path not empty:", int(fire_df["ann_not_empty"].sum()))
    print("fire annotation_path exists:", int(fire_df["ann_exists"].sum()))

    if "has_fire_box" in fire_df.columns:
        print("has_fire_box=True:", int((fire_df["has_fire_box"] == True).sum()))

    if "bbox_count" in fire_df.columns:
        print("bbox_count > 0:", int((fire_df["bbox_count"].fillna(0).astype(int) > 0).sum()))

    print("\n无 bbox 的 fire 样例:")
    cols = [
        c for c in [
            "filename",
            "source_filename",
            "label",
            "annotation_path",
            "has_fire_box",
            "bbox_count",
            "annotation_size_match",
            "source_type",
            "aug_type"
        ]
        if c in fire_df.columns
    ]

    print(
        fire_df[
            ~fire_df["ann_exists"]
        ][cols].head(20)
    )

    return fire_df


for ensemble_id in range(N_ENSEMBLE):
    train_csv = find_train_csv(ensemble_id)
    _ = check_bbox_status(train_csv, f"train ensemble {ensemble_id:02d}")

_ = check_bbox_status(VAL_CSV, "val")
_ = check_bbox_status(TEST_CSV, "test")


train ensemble 00
E:\Programming\Python\DeepLearning\比赛\fire_splits\ensemble_splits\ensemble_train_00.csv
total: 1000
label distribution:
label
0    500
1    500
Name: count, dtype: int64

fire total: 500
fire annotation_path not empty: 500
fire annotation_path exists: 500
has_fire_box=True: 499
bbox_count > 0: 500

无 bbox 的 fire 样例:
Empty DataFrame
Columns: [filename, source_filename, label, annotation_path, has_fire_box, bbox_count, annotation_size_match, source_type, aug_type]
Index: []

val
E:\Programming\Python\DeepLearning\比赛\fire_splits\val_fixed_real_only.csv
total: 100
label distribution:
label
0    50
1    50
Name: count, dtype: int64

fire total: 50
fire annotation_path not empty: 50
fire annotation_path exists: 50
has_fire_box=True: 50
bbox_count > 0: 50

无 bbox 的 fire 样例:
Empty DataFrame
Columns: [filename, source_filename, label, annotation_path, has_fire_box, bbox_count, annotation_size_match, source_type, aug_type]
Index: []

test
E:\Programming\Python\DeepLearning\比赛\

In [ ]:
# =============
# 可选：只从 latest 断点恢复
# =============
# 训练意外中断时使用这个 cell。
# 平时不要和“初次训练”“继续训练”一起 Run All。

# RESUME_FROM = "latest"
# RESUME_END_EPOCH = 15
#
# state_resume = run_training(
#     resume_training=True,
#     resume_from=RESUME_FROM,
#     end_epoch=RESUME_END_EPOCH,
#     reset_optimizer=False,
#     reset_scheduler=False,
#     lr_head=LR_HEAD,
#     lr_backbone=LR_BACKBONE,
#     run_name="resume_from_latest_global_local_attention_supervised",
#     show_step=True,
#     step_log_interval=10
# )

In [ ]:
# =============
# 继续训练
# =============
# 用于从 latest / best / best_val_loss / final 权重继续训练。
#
# 推荐：
# - 训练中断：CONTINUE_FROM = "latest"，reset_optimizer=False
# - 从 best 微调：CONTINUE_FROM = "best"，reset_optimizer=True
# - 如果 false positive 高且概率过激，可以试 CONTINUE_FROM = "best_val_loss"

CONTINUE_FROM = "best"
CONTINUE_END_EPOCH = 25

CONTINUE_LR_HEAD = 2e-6
CONTINUE_LR_BACKBONE = 2e-8

state_continue = run_training(
    resume_training=True,
    resume_from=CONTINUE_FROM,
    end_epoch=CONTINUE_END_EPOCH,
    reset_optimizer=True,
    reset_scheduler=True,
    lr_head=CONTINUE_LR_HEAD,
    lr_backbone=CONTINUE_LR_BACKBONE,
    run_name=f"continue_from_{CONTINUE_FROM}_attention_supervised",
    show_step=True,
    step_log_interval=10
)